# 07 · Checks and export

Runs the data checks first (unique keys, no orphan keys, silver Σ units = gold Σ units). If anything fails,
nothing is exported. Then every gold table is written as Parquet to the `gold.export` volume, for the local
reconciliation against Project 1 and, if the Power BI connector test failed, for upload to OneLake.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath("../src"))  # the repo's src/, via the Databricks Git folder

from favorita_spark.config import Config

cfg = Config()
spark.sql(f"USE CATALOG {cfg.catalog}")

In [ ]:
from favorita_spark.checks import run_checks

failures = run_checks(spark)
if failures:
    raise RuntimeError(f"Data checks failed, nothing exported: {failures}")
print("All data checks pass.")

In [ ]:
TABLES = ["dim_date", "stg_store", "stg_item", "stg_store_day", "fact_sales", "fact_stockout_risk",
          "fact_stockout_run", "fact_promo_event", "promo_family_summary", "stockout_bh_summary"]
for table in TABLES:
    spark.table(f"gold.{table}").write.mode("overwrite").parquet(f"/Volumes/{cfg.catalog}/gold/export/{table}")
display(dbutils.fs.ls(f"/Volumes/{cfg.catalog}/gold/export"))

## Next, on your machine

```bash
databricks fs cp -r dbfs:/Volumes/workspace/gold/export data/export --overwrite
python scripts/reconcile.py --export data/export --flagship ../Flagship/data/full/gold
```

**OneLake fallback:** upload the `data/export/<table>` folders to the Project 1 lakehouse (Files), then
*Load to Tables*, or shortcut them.